In [2]:

import uproot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from catboost import CatBoostClassifier, Pool


import time
from IPython import display

import matplotlib
matplotlib.rcParams.update(matplotlib.rcParamsDefault)

# import subprocess
# data_dir = "/l/izaac/data/24collision"
# result = subprocess.run(['ls', data_dir], stdout=subprocess.PIPE, text=True)
# file_list = [data_dir+"/"+file_path+":Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree" for file_path in result.stdout.split('\n') if file_path]

import subprocess
data_dir = "/l/izaac/data/24collision-magup"
result = subprocess.run(['ls', data_dir], stdout=subprocess.PIPE, text=True)
file_list = [data_dir+"/"+file_path+":Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree" for file_path in result.stdout.split('\n') if file_path]

# cut = '(mup_PID_MU > 0) & (mum_PID_MU > 0) & (Jpsi_MAXDOCA < 0.15) & (p_PID_P > 0) & ( p_P > 10000)  &(L_P > 10000)& ( pi_P > 3000) & (L_PT > 450) & (Lb_CHI2 < 50)  & (Jpsi_P > 18000) & (L_MASS < 1400) & (L_MASS > 950) & (Jpsi_CHI2 < 3 ) & (Jpsi_BPVDIRA > 0.999) &  (L_CHI2 < 50) & (p_MINIP < 80) & (L_END_VZ > 5000) & (Lb_MINIPCHI2 < 20) '
# cut = '(Jpsi_P > 10000) & (mup_PID_MU > 0) & (mum_PID_MU > 0) & (Jpsi_MAXDOCA < 0.15) & (p_PID_P > 0) & ( p_P > 10000)& ( p_PT > 400) & ( pi_P > 2000) & (L_PT > 450) & (Lb_CHI2 < 150) & (L_MASS < 1500) & (L_MASS > 950) & (Jpsi_CHI2 < 4 ) &  (L_CHI2 < 100) & (L_END_VZ > 5000) '
preselection_cut = '(Jpsi_P > 10000) & (mup_PID_MU > 0) & (mum_PID_MU > 0) & (Jpsi_MAXDOCA < 0.15) & (p_PID_P > 0) & ( p_P > 10000)& ( p_PT > 400) & ( pim_P > 2000) & (L_PT > 450) & (Lb_CHI2 < 150) & (Jpsi_CHI2 < 4 ) &  (L_CHI2 < 100) & (L_MASS < 1700) '
L_cut = '(L_MASS > 1300)'
L_key = ['L_MASS']

keys = ["L_MASS",'Lb_MASS','Jpsi_MASS','Jpsi_P', "Jpsi_PT",'Jpsi_ETA','Jpsi_CHI2','Jpsi_MAXDOCA','Jpsi_BPVDIRA','Jpsi_END_VRHO','Jpsi_END_VX','Jpsi_END_VY','Jpsi_END_VZ','Jpsi_BPVIP','Jpsi_MINIPCHI2','L_P','L_PT','L_PX','L_PY','L_PZ','L_CHI2',
               'L_END_VZ','L_END_VX','L_END_VY',"L_END_VRHO","L_BPVDIRA","L_BPVIP","L_BPVIPCHI2",'L_BPVFDCHI2','L_MINIPCHI2',"Lb_P","Lb_PT",'Lb_PX','Lb_PY','Lb_PZ',"Lb_CHI2","Lb_MAXDOCA","Lb_BPVDIRA","Lb_BPVVDRHO",
               "Lb_BPVIP",'Lb_BPVIPCHI2','Lb_BPVFDCHI2','Lb_MINIPCHI2','Lb_ETA',"p_P","p_PT", 'p_ETA','p_PX','p_PY','p_PZ',"p_PID_P",
               "p_PID_K","p_MINIP",'p_GHOSTPROB',"pim_P", 'pim_ETA',"pim_PT",'pim_PX','pim_PY','pim_PZ','pim_GHOSTPROB','mum_P','mum_PT','mum_PX','mum_PY','mum_PZ','mum_CHI2','mum_GHOSTPROB','mum_PID_P','mum_PID_K','mum_PID_MU',
               'mum_PID_E','mum_MINIP','mum_MINIPCHI2','mup_P','mup_PT','mup_PX','mup_PY','mup_PZ','mup_CHI2','mup_GHOSTPROB','mup_PID_P','mup_PID_K','mup_PID_MU','mup_PID_E','mup_MINIP','mup_MINIPCHI2',
               'totCandidates','p_QOVERP', 'pim_QOVERP', 'Lb_END_VX','Lb_END_VY','Lb_END_VZ','L_ENERGY','p_ENERGY','pim_ENERGY', 'p_TRACK_POS_CLOSESTTOBEAM_X','p_TRACK_POS_CLOSESTTOBEAM_Y','p_TRACK_POS_CLOSESTTOBEAM_Z','pim_TRACK_POS_CLOSESTTOBEAM_X','pim_TRACK_POS_CLOSESTTOBEAM_Y','pim_TRACK_POS_CLOSESTTOBEAM_Z',
               'EVENTNUMBER'] #for magDown

keys2 = ["L_MASS",'Lb_MASS','Jpsi_MASS','Jpsi_P', "Jpsi_PT",'Jpsi_ETA','Jpsi_CHI2','Jpsi_MAXDOCA','Jpsi_BPVDIRA','Jpsi_END_VRHO','Jpsi_END_VX','Jpsi_END_VY','Jpsi_END_VZ','Jpsi_BPVIP','Jpsi_MINIPCHI2','L_P','L_PT','L_PX','L_PY','L_PZ','L_CHI2',
               'L_END_VZ','L_END_VX','L_END_VY',"L_END_VRHO","L_BPVDIRA","L_BPVIP","L_BPVIPCHI2",'L_BPVFDCHI2','L_MINIPCHI2',"Lb_P","Lb_PT",'Lb_PX','Lb_PY','Lb_PZ',"Lb_CHI2","Lb_MAXDOCA","Lb_BPVDIRA","Lb_BPVVDRHO",
               "Lb_BPVIP",'Lb_BPVIPCHI2','Lb_BPVFDCHI2','Lb_MINIPCHI2','Lb_ETA',"p_P","p_PT", 'p_ETA','p_PX','p_PY','p_PZ',"p_PID_P",
               "p_PID_K","p_MINIP",'p_GHOSTPROB',"pim_P", 'pim_ETA',"pim_PT",'pim_PX','pim_PY','pim_PZ','pim_GHOSTPROB','mum_P','mum_PT','mum_PX','mum_PY','mum_PZ','mum_CHI2','mum_GHOSTPROB','mum_PID_P','mum_PID_K','mum_PID_MU',
               'mum_PID_E','mum_MINIP','mum_MINIPCHI2','mup_P','mup_PT','mup_PX','mup_PY','mup_PZ','mup_CHI2','mup_GHOSTPROB','mup_PID_P','mup_PID_K','mup_PID_MU','mup_PID_E','mup_MINIP','mup_MINIPCHI2',
               'totCandidates','p_QOVERP', 'pim_QOVERP', 'Lb_END_VX','Lb_END_VY','Lb_END_VZ','L_ENERGY','p_ENERGY','pim_ENERGY', 'p_REFERENCEPOINT_X','p_REFERENCEPOINT_Y','p_REFERENCEPOINT_Z','pim_REFERENCEPOINT_X','pim_REFERENCEPOINT_Y','pim_REFERENCEPOINT_Z',
               'EVENTNUMBER','p_TX','p_TY','pim_TX','pim_TY'] #for magUp ,'p_POS_COV_MATRIX','pim_POS_COV_MATRIX', 'Lb_DTF_M', 'Lb_DTF_PV_M','Lb_DTF_FixJpsi_M' ,'Lb_DTF_PV_FixJpsiLmd_M','Lb_DTF_PV_FixJpsi_M'

training_keys = ['Jpsi_PT','Jpsi_CHI2', 'L_END_VRHO', 'L_BPVDIRA', 'L_BPVIP', 'L_BPVIPCHI2','Lb_MASS',
       'Lb_BPVDIRA', 'Lb_BPVIP', 'Lb_BPVVDRHO', 'Lb_MAXDOCA', 'Lb_P', 'Lb_PT',
       'Lb_CHI2', 'p_PID_P', 'p_MINIP', 'mup_PID_MU', 'L_PT', 'Jpsi_MAXDOCA',
       'Lb_MINIPCHI2', 'L_P', 'Jpsi_BPVDIRA', 'L_CHI2', 'Lb_ETA', 'L_END_VZ',
       'L_END_VX', 'L_END_VY', 'Jpsi_ETA', 'p_P', 'p_PT', 'p_ETA',
       'p_GHOSTPROB', 'pim_P', 'pim_PT', 'pim_ETA', 'pim_GHOSTPROB','EVENTNUMBER']+ ['p_PX'] + ['p_PY'] + ['p_PZ'] + ['L_PX'] + ['L_PY'] + ['L_PZ'] + ['Lb_PX'] + ['Lb_PY'] + ['Lb_PZ']



data24 = pd.DataFrame(columns=keys2)


# for file in file_list:  #para utilizar todas las tuplas disponibles
for file in file_list[:50]:  #para utilizar los 50 primeros archivos 
    print("Now opening",file)

    with uproot.open(file) as tuple_file:
        # data24= data24.merge(tuple_file.arrays(expressions=keys,library='pd',cut=preselection_cut),how="outer") ##for magDown 
        data24 = pd.concat([data24,tuple_file.arrays(expressions=keys2,library='pd',cut=preselection_cut)],axis=0) # for magUp cause cov_matrix need concat insted of merge

data24.to_pickle('data24_magup_50.pkl')

Now opening /l/izaac/data/24collision-magup/00229346_00000001_1.data_turbopass_bandq_lb2jpsilmd.root:Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree


/tmp/javeser/ipykernel_11391/2610032884.py:64: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  data24 = pd.concat([data24,tuple_file.arrays(expressions=keys2,library='pd',cut=preselection_cut)],axis=0) # for magUp cause cov_matrix need concat insted of merge


Now opening /l/izaac/data/24collision-magup/00229346_00000002_1.data_turbopass_bandq_lb2jpsilmd.root:Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree
Now opening /l/izaac/data/24collision-magup/00229346_00000003_1.data_turbopass_bandq_lb2jpsilmd.root:Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree
Now opening /l/izaac/data/24collision-magup/00229346_00000004_1.data_turbopass_bandq_lb2jpsilmd.root:Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree
Now opening /l/izaac/data/24collision-magup/00229346_00000005_1.data_turbopass_bandq_lb2jpsilmd.root:Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree
Now opening /l/izaac/data/24collision-magup/00229346_00000006_1.data_turbopass_bandq_lb2jpsilmd.root:Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree
Now opening /l/izaac/data/24collision-magup/00229346_00000007_1.data_turbopass_bandq_lb2jpsilmd.root:Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree
Now opening /l/izaac/data/24collision-magup/00229346_00000008_1.data_turbopass_bandq_lb2jpsilmd.root:Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree
Now opening /l/izaac/data/24collision-mag

Para cargar todas las varaibles disponibles de una tupla

In [1]:
import subprocess
import uproot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from catboost import CatBoostClassifier, Pool


import time
from IPython import display

import matplotlib
#"/l/izaac/data/24collision
data_dir = "/l/izaac/data/24collision"
result = subprocess.run(['ls', data_dir], stdout=subprocess.PIPE, text=True)
file_list = [data_dir + "/" + file_path + ":Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree" for file_path in result.stdout.split('\n') if file_path]

# Lista para almacenar los DataFrames temporales
data_frames = []

# Cargar las primeras 50 tuplas
for file in file_list[:1]:
    print("Now opening", file)

    with uproot.open(file) as tuple_file:
        # Obtener todas las variables disponibles en la tupla
        all_keys = tuple_file.keys()

        # Cargar todas las variables en un DataFrame temporal
        # data_temp = tuple_file.arrays(expressions=all_keys, library='pd')

        # # Aplicar el preselection_cut si es necesario
        # data_temp = data_temp.query(preselection_cut)

        # data_frames.append(data_temp)

# # Concatenar todos los DataFrames en uno solo
# data24 = pd.concat(data_frames, ignore_index=True)
print(all_keys)

# Guardar el DataFrame resultante en un archivo pickle
# data24.to_pickle('data24_magup_keys.pkl')


Now opening /l/izaac/data/24collision/00222479_00000001_1.data_turbopass_bandq_lb2jpsilmd.root:Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree
['totCandidates', 'nCandidate', 'indx', 'ALLPVX', 'ALLPVY', 'ALLPVZ', 'BUNCHCROSSING_ID', 'BUNCHCROSSING_TYPE', 'EVENTNUMBER', 'EVENTTYPE', 'GPSTIME', 'Hlt1D2KKDecision', 'Hlt1D2KPiDecision', 'Hlt1D2PiPiDecision', 'Hlt1DetJpsiToMuMuNegTagLineDecision', 'Hlt1DetJpsiToMuMuPosTagLineDecision', 'Hlt1DiElectronDisplacedDecision', 'Hlt1DiElectronHighMassDecision', 'Hlt1DiElectronHighMass_SSDecision', 'Hlt1DiMuonDrellYanDecision', 'Hlt1DiMuonDrellYan_SSDecision', 'Hlt1DiMuonDrellYan_VLowMassDecision', 'Hlt1DiMuonDrellYan_VLowMass_SSDecision', 'Hlt1DiMuonHighMassDecision', 'Hlt1DiMuonLowMassDecision', 'Hlt1DiMuonNoIPDecision', 'Hlt1DiMuonNoIP_SSDecision', 'Hlt1DiMuonSoftDecision', 'Hlt1DiPhotonHighMassDecision', 'Hlt1KsToPiPiDecision', 'Hlt1LowPtDiMuonDecision', 'Hlt1LowPtMuonDecision', 'Hlt1Pi02GammaGammaDecision', 'Hlt1SingleHighEtDecision', 'Hlt1SingleHighPtElec